# 139. Word Break
**Difficulty:** 🟡 Medium · **Topic:** Dynamic Programming · **LeetCode:** https://leetcode.com/problems/word-break/

## 💡 Concepts

**Core concept(s):** DP over prefixes — `dp[i]` = can the first `i` characters be split into dictionary words?

**Why it applies here:** A string is breakable if some dictionary word ends at the current position **and** the part before that word is itself breakable. That reuses answers for shorter prefixes, so we fill `dp` left to right.

**Key intuition:** The first `i` letters work if a word ends at `i` and the letters before that word also work.

---

### 📚 What is Dynamic Programming (DP)?
**DP** solves a big problem by solving smaller **overlapping** subproblems once and reusing the answers. Two styles: **memoization** (recursion that caches results) and **tabulation** (fill a table from the smallest cases up).
- **Why it's fast:** it turns exponential re-computation into a single sweep over the subproblems.
- **In Python:** a `dict`/list cache, or a `dp` list/2-D table.

### 📚 Subproblems & Recurrence
The heart of DP is a **recurrence**: the answer for a state written in terms of smaller states (e.g. `dp[i] = dp[i-1] + dp[i-2]`). Find the recurrence and the base cases, and the code writes itself.

---

**Prerequisite knowledge:**
- Prefix DP.
- A set for O(1) word lookups.

## 📝 Problem

Given a string `s` and a dictionary, return `True` if `s` can be split into a sequence of dictionary words.

**Example**
```
s="leetcode", words=["leet","code"] -> True
s="catsandog", words=["cats","dog","sand","and","cat"] -> False
```

> Two approaches: exponential brute force and `O(n²)` DP.

### Approach 1 — Brute Recursion (worst)

**Idea:** Try every prefix that is a word, then recurse on the rest.

**Time:** exponential. **Space:** `O(n)`.

In [ ]:
def word_break_brute(s, words):
    wordset = set(words)                   # O(1) word lookups
    def dfs(start):                        # can s[start:] be split into words?
        if start == len(s):
            return True                    # consumed the whole string
        for end in range(start + 1, len(s) + 1):
            if s[start:end] in wordset and dfs(end):   # a word here, and the rest also splits
                return True
        return False
    return dfs(0)

### Approach 2 — Prefix DP (optimal)

**Idea:** `dp[i]` = first `i` chars breakable. `dp[i]` is true if some `j < i` has `dp[j]` true and `s[j:i]` is a word.

**Time:** `O(n²)` (times word-length for the slice/lookup). **Space:** `O(n)`.

In [ ]:
def word_break_dp(s, words):
    wordset = set(words)
    n = len(s)
    dp = [False] * (n + 1)                 # dp[i] = can the first i characters be split?
    dp[0] = True                           # empty prefix is trivially splittable
    for i in range(1, n + 1):
        for j in range(i):                 # try every place a final word could start
            if dp[j] and s[j:i] in wordset:# the part before splits AND s[j:i] is a word
                dp[i] = True
                break
    return dp[n]

In [ ]:
# Correctness check
tests = [
    ("leetcode",["leet","code"],True),
    ("applepenapple",["apple","pen"],True),
    ("catsandog",["cats","dog","sand","and","cat"],False),
]
for s, words, exp in tests:
    a, b = word_break_brute(s, words), word_break_dp(s, words)
    print(f"{s!r} -> brute={a}, dp={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |

Inputs are shaped to force the worst case. (Exponential brute-force versions are shown in the code but omitted from timing where they would blow up — noted per notebook.)

*(Brute force is omitted from timing.)*

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    s = 'a' * n            # breakable many ways -> full DP scan
    words = ['a', 'aa', 'aaa']
    return (s, words)
solutions = {
    "dp O(n^2)": word_break_dp,
}
sizes = [200, 400, 800, 1600]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Prefix DP:** "can the first i characters be formed?" reused to build longer prefixes.
- **Set for fast membership:** dictionary lookups in O(1).
- **Signal:** "can the string be split / segmented into pieces from a set".
- **Related problems:** Word Break II (list them), Concatenated Words, Palindrome Partitioning.
- **Common pitfalls:** (1) exponential recursion without memo; (2) forgetting `dp[0] = True`.